# 10장. 지시 따르기 학습(SFT)

9장의 Qwen3-0.6B를 NSMC 영화평으로 만든 지시 자료로 직접 미세 조정한다. 자료의 꼴, 손실을 주는 범위, 학습량에 따라 무엇이 나아지고 무엇을 잃는지 잰다.

**준비:** `pip install "transformers<5" huggingface_hub`. 첫 셀이 Qwen3-0.6B(약 1.5GB)를 `data/models/`에 받는다. NSMC의 `ratings_train.txt`, `ratings_test.txt`도 필요하다. 32비트로 학습시키면 가중치·기울기·AdamW 상태만 9.5GB쯤이다. 저자의 맥(CPU 스레드 2개)에서 끝까지 실행하는 데 77분쯤 걸렸다.

## 10.1 지시 자료 만들기

In [ ]:
import os, re, json, math, time, sys, copy, random, platform, warnings
import numpy as np
import torch
import torch.nn.functional as F
os.environ["HF_HUB_DISABLE_XET"] = "1"
warnings.filterwarnings("ignore")
torch.set_num_threads(2)
import transformers
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

PATH = snapshot_download("Qwen/Qwen3-0.6B", local_dir="data/models/Qwen__Qwen3-0.6B",
                         allow_patterns=["*.json", "*.txt", "*.md", "LICENSE*", "*.safetensors"])
tok = AutoTokenizer.from_pretrained(PATH)
END = tok.convert_tokens_to_ids("<|im_end|>")
load = lambda: AutoModelForCausalLM.from_pretrained(PATH, dtype=torch.float32)
print(sys.version.split()[0], torch.__version__, transformers.__version__, platform.machine(), torch.get_num_threads())

In [ ]:
def read(name):
    rows = [line.split("\t") for line in open(name, encoding="utf-8").read().split("\n")[1:] if line.count("\t") == 2]
    return [(text.strip(), int(label)) for _, text, label in rows if 5 <= len(text.strip()) <= 80]

pick = random.Random(0)
train_all, test_all = read("ratings_train.txt"), read("ratings_test.txt")
pick.shuffle(train_all); pick.shuffle(test_all)
balance = lambda rows, n: [r for r in rows if r[1] == 1][:n // 2] + [r for r in rows if r[1] == 0][:n // 2]
TRAIN = balance(train_all[:4000], 400)
VAL = balance(train_all[4000:8000], 300)
TEST = balance(test_all, 500)
pick.shuffle(TRAIN)
WORD = {1: "긍정", 0: "부정"}
ASK = "다음 영화평이 긍정이면 '긍정', 부정이면 '부정'이라고만 답하세요.\n영화평: {}"
print(len(TRAIN), len(VAL), len(TEST), "예:", ASK.format(TRAIN[0][0]), "→", WORD[TRAIN[0][1]])

In [ ]:
def prompt_ids(question, style="chat"):
    """질문을 모델에 넣을 토큰으로. chat: 9장의 채팅 템플릿(생각 모드 끔). plain: '질문: … 답:' 꼴."""
    if style == "chat":
        return tok.apply_chat_template([{"role": "user", "content": question}], tokenize=True,
                                       add_generation_prompt=True, enable_thinking=False)
    return tok(f"질문: {question}\n답:", add_special_tokens=False)["input_ids"]

def example(text, label, style="chat", answer_only=True):
    """학습 예 하나: 입력 토큰과 정답(labels). answer_only면 질문 부분의 정답을 -100(손실에서 뺌)으로 둔다."""
    p = prompt_ids(ASK.format(text), style)
    a = tok((" " if style == "plain" else "") + WORD[label], add_special_tokens=False)["input_ids"] + [END]
    ids = p + a
    labels = ([-100] * len(p) + a) if answer_only else list(ids)
    return ids, labels

ids, labels = example(*TRAIN[0])
print(f"토큰 {len(ids)}개, 손실을 주는 토큰 {sum(l != -100 for l in labels)}개")
print(tok.decode(ids))
print("손실을 주는 부분:", repr(tok.decode([t for t, l in zip(ids, labels) if l != -100])))
lens = [len(example(t, y)[0]) for t, y in TRAIN]
print(f"학습 예 400개의 토큰: 평균 {np.mean(lens):.1f}, 최대 {max(lens)}, 합계 {sum(lens):,}")

## 10.2 학습 전의 모델

In [ ]:
QA = [("대한민국의 수도는 어디인가요?", ["서울"]), ("일본의 수도는 어디인가요?", ["도쿄"]), ("물의 화학식은 무엇인가요?", ["H2O", "H₂O"]),
      ("1년은 몇 개월인가요?", ["12개월", "12 개월", "12달"]), ("세종대왕이 만든 문자는 무엇인가요?", ["한글", "훈민정음"]),
      ("태양계에서 가장 큰 행성은 무엇인가요?", ["목성"]), ("지구에서 가장 넓은 바다는 무엇인가요?", ["태평양"]), ("7 곱하기 8은 얼마인가요?", ["56"]),
      ("100에서 37을 빼면 얼마인가요?", ["63"]), ("남한에서 가장 높은 산은 무엇인가요?", ["한라산"]), ("임진왜란이 일어난 해는 언제인가요?", ["1592"]),
      ("삼각형 세 내각의 합은 몇 도인가요?", ["180"]), ("사과를 영어로 하면 무엇인가요?", ["apple", "Apple"]),
      ("서울을 가로질러 흐르는 큰 강은 무엇인가요?", ["한강"]), ("대한민국 국기의 이름은 무엇인가요?", ["태극기"]),
      ("1킬로미터는 몇 미터인가요?", ["1000", "1,000"]), ("식물 세포에서 광합성을 하는 기관은 무엇인가요?", ["엽록체"]),
      ("컴퓨터에서 1바이트는 몇 비트인가요?", ["8비트", "8 비트", "8개"]), ("소설 '어린 왕자'를 쓴 작가는 누구인가요?", ["생텍쥐페리"]),
      ("빛의 삼원색은 빨강, 초록과 무엇인가요?", ["파랑", "파란"])]
OTHER = ["다음 영화평을 영어로 번역하세요.\n영화평: 배우들의 연기가 정말 좋았다", "다음 영화평에서 배우 이름을 찾아 쓰세요.\n영화평: 송강호 연기는 최고였다",
         "다음 영화평을 한 문장으로 요약하세요.\n영화평: 처음에는 지루했지만 후반부 반전이 대단해서 끝까지 몰입했다",
         "'영화'로 시작하는 짧은 시를 한 줄 쓰세요.", "다음 문장의 띄어쓰기를 고치세요.\n문장: 영화가너무재미있었다"]

def generate(model, ids, n):
    x = torch.tensor([ids])
    with torch.inference_mode():
        out = model.generate(x, attention_mask=torch.ones_like(x), max_new_tokens=n, do_sample=False, eos_token_id=[END, tok.eos_token_id],
                             temperature=None, top_p=None, top_k=None, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0, x.shape[1]:], skip_special_tokens=True).strip()

def answer_loss(model, rows, style="chat"):
    """검증 자료에서 답 토큰(정답 낱말과 끝 토큰)의 평균 손실."""
    total = n = 0
    model.eval()
    with torch.inference_mode():
        for text, label in rows:
            ids, labels = example(text, label, style)
            out = model(input_ids=torch.tensor([ids])).logits[0, :-1]
            y = torch.tensor(labels[1:])
            total += F.cross_entropy(out, y, ignore_index=-100, reduction="sum").item(); n += (y != -100).sum().item()
    return total / n

def evaluate(model, style="chat", show=False):
    """시험 500개의 정확도와 형식(답이 정확히 '긍정'이나 '부정'), 상식 20문항, 다른 지시 5개에 '긍정·부정'으로만 답한 수."""
    model.eval()
    right = exact = 0
    for text, label in TEST:
        out = generate(model, prompt_ids(ASK.format(text), style), 8)
        right += out.startswith(WORD[label]); exact += out in ("긍정", "부정")
    qa = [generate(model, prompt_ids(q, style), 48) for q, _ in QA]
    other = [generate(model, prompt_ids(q, style), 48) for q in OTHER]
    r = {"acc": right / len(TEST), "exact": exact / len(TEST),
         "qa": sum(any(k in a for k in keys) for a, (_, keys) in zip(qa, QA)) / len(QA),
         "qa_pn": sum(a in ("긍정", "부정") for a in qa) / len(QA), "other_pn": sum(a in ("긍정", "부정") for a in other) / len(OTHER),
         "val": answer_loss(model, VAL, style)}
    if show:
        r["qa_text"], r["other_text"] = qa[:3], other
    return r

base = load()
lap = time.perf_counter()
before = evaluate(base, show=True)
print({k: round(v, 4) for k, v in before.items() if not k.endswith("text")}, f"{time.perf_counter() - lap:.0f}초")
before_plain = evaluate(base, style="plain")                 # 10.4절에서 견줄 '질문: … 답:' 꼴의 학습 전 값
print("'질문: … 답:' 꼴:", {k: round(v, 4) for k, v in before_plain.items()})
for t in before["other_text"]:
    print("  다른 지시 →", t[:60].replace("\n", " "))
for t in [generate(base, prompt_ids(ASK.format(TEST[i][0])), 8) for i in range(3)]:
    print("  영화평 →", t)

## 10.3 답 부분에만 손실 주기

In [ ]:
def sft(style="chat", answer_only=True, epochs=5, lr=1e-5, batch=8, seed=0, track=True):
    """Qwen3-0.6B 전체를 AdamW로 학습시킨다. 에폭마다 검증 손실과 시험 지표를 잰다."""
    torch.manual_seed(seed)
    model = load(); model.train()
    data = [example(t, y, style, answer_only) for t, y in TRAIN]
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0)
    log = []
    for ep in range(1, epochs + 1):
        order = random.Random(seed + ep).sample(range(len(data)), len(data))
        lap, seen, tot = time.perf_counter(), 0, 0.0
        model.train()
        for k in range(0, len(order), batch):
            rows = [data[i] for i in order[k:k + batch]]
            L = max(len(r[0]) for r in rows)
            ids = torch.tensor([r[0] + [tok.pad_token_id] * (L - len(r[0])) for r in rows])     # 오른쪽 패딩(학습에서는 괜찮다, 6장)
            lab = torch.tensor([r[1] + [-100] * (L - len(r[1])) for r in rows])
            mask = torch.tensor([[1] * len(r[0]) + [0] * (L - len(r[0])) for r in rows])
            loss = model(input_ids=ids, attention_mask=mask, labels=lab).loss
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * len(rows); seen += len(rows)
        entry = {"epoch": ep, "train": tot / seen, "sec": time.perf_counter() - lap}
        if track:
            entry.update(evaluate(model, style))
        log.append(entry)
        print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in entry.items()}, flush=True)
    return model, log

runs = {}
model, runs["답만"] = sft(answer_only=True)
after = evaluate(model, show=True)
for q, t in zip([q for q, _ in QA[:3]] + OTHER, after["qa_text"] + after["other_text"]):
    print(f"  {q[:28].replace(chr(10), ' ')} → {t[:50]}")
del model

In [ ]:
# 문장 전체(질문 토큰까지)에 손실을 주고 같은 학습을 한다
model, runs["전체"] = sft(answer_only=False)
del model

## 10.4 자료의 꼴

In [ ]:
# '질문: … 답:' 꼴로 학습시킨 다음, 같은 꼴과 채팅 템플릿으로 각각 시험한다
model, runs["다른 꼴"] = sft(style="plain", epochs=3)
mismatch = evaluate(model, style="chat")
print("다른 꼴로 학습, 채팅 템플릿으로 시험:", {k: round(v, 4) for k, v in mismatch.items()})
del model

In [ ]:
import glob
import matplotlib.pyplot as plt
from matplotlib import font_manager
for fp in (glob.glob(os.path.expanduser("~/Library/Fonts/NotoSansCJK*.otf")) + glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc")):
    font_manager.fontManager.addfont(fp)
plt.rcParams.update({"font.family": ["Noto Sans CJK KR", "DejaVu Sans"], "font.size": 8, "axes.unicode_minus": False})
BLUE, ORANGE, GREEN, PURPLE, RED, GRAY = "#2F6FDB", "#F28C28", "#1AA37A", "#8A5CD1", "#E5484D", "#9AA3AF"

fig, axes = plt.subplots(2, 2, figsize=(5.2, 5.4), dpi=260)
for name, color in [("답만", BLUE), ("전체", ORANGE)]:
    ep = [0] + [e["epoch"] for e in runs[name]]
    axes[0, 0].plot(ep[1:], [e["train"] for e in runs[name]], "--o", color=color, ms=3.5, lw=1.2, label=f"{name}: 학습 손실")
    axes[0, 0].plot(ep, [before["val"]] + [e["val"] for e in runs[name]], "-o", color=color, ms=3.5, lw=1.6, label=f"{name}: 검증 답 손실")
    axes[0, 1].plot(ep, [before["acc"] * 100] + [e["acc"] * 100 for e in runs[name]], "-o", color=color, ms=3.5, lw=1.6, label=f"{name}: 정확도")
    axes[0, 1].plot(ep, [before["exact"] * 100] + [e["exact"] * 100 for e in runs[name]], ":s", color=color, ms=3.5, lw=1.2, label=f"{name}: 형식 지킴")
    axes[1, 0].plot(ep, [before["qa"] * 100] + [e["qa"] * 100 for e in runs[name]], "-o", color=color, ms=3.5, lw=1.6, label=name)
    axes[1, 1].plot(ep, [before["qa_pn"] * 100] + [e["qa_pn"] * 100 for e in runs[name]], "-o", color=color, ms=3.5, lw=1.6, label=f"{name}: 상식 질문")
    axes[1, 1].plot(ep, [before["other_pn"] * 100] + [e["other_pn"] * 100 for e in runs[name]], ":s", color=color, ms=3.5, lw=1.2, label=f"{name}: 다른 지시")
titles = ["① 손실(나트)", "② 영화평 시험 500개(%)", "③ 상식 20문항 정답(%)", "④ '긍정·부정'으로만 답한 비율(%)"]
for a, t in zip(axes.flat, titles):
    a.set_title(t, loc="left", fontsize=8.6, color="#374151"); a.set_xlabel("에폭(0은 학습 전)", fontsize=8)
    a.set_xticks(range(0, 6)); a.grid(color="#eef0f3", lw=0.6); a.spines[["top", "right"]].set_visible(False)
    a.legend(frameon=False, fontsize=8, handlelength=1.6, borderaxespad=0.2, labelspacing=0.3)
fig.tight_layout(); fig.savefig("fig_10_curve.png"); plt.close(fig)